# FT 3B Solo — ARC-Challenge Direct Answer Fine-Tuning

**Purpose:** Fine-tune Qwen2.5-3B-Instruct to answer ARC-Challenge 4-choice science MCQ questions **directly**. Evaluate on N=900 at seed=42 — the same split used in the paper.

ARC-Challenge is Tier-2 in the paper taxonomy: CoT alone achieves 80.0% = Guided.
The key question is whether FT 3B Solo matches or exceeds both at 3.5× less compute.

| Condition | Compute | Paper result |
|---|---|---|
| Baseline (1.5B×5) | 7.5B pp | 71.7% |
| CoT (1.5B×5) | 7.5B pp | 80.0% |
| **FT 3B Solo (this run)** | **3.0B pp** | **TBD** |
| Guided pipeline | 10.5B pp | 80.0% |

Random chance = 25% (4-choice MCQ).

> Seed=42 · N=900 · Same question split as original paper run

In [1]:
# CELL 1 — Install
# !pip install -q transformers==4.44.0
# !pip install -q accelerate==0.33.0
# !pip install -q peft==0.12.0
# !pip install -q datasets==2.20.0
# !pip install -q trl==0.9.6
# !pip install -q huggingface_hub
!pip install trl
print("Done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 10.2 MB/s eta 0:00:00 0:00:01
Done.


In [ ]:
# CELL 2 — Login
from huggingface_hub import login
login("")  # paste your HF token
print("Login done")

Login done


In [3]:
# CELL 3 — Imports
import os, json, re, random, time
import torch
from collections import Counter
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer
from tqdm.notebook import tqdm

OUTPUT_DIR = "/content/arc_ft3b_solo"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

GPU: Tesla T4
VRAM: 15.6 GB


In [4]:
# CELL 4 — Config
# eval_seed=42, eval_n=900 MUST match paper exactly.
CONFIG = {
    "model_name"        : "microsoft/Phi-3.5-mini-instruct",
    "dataset_name"      : "allenai/ai2_arc",
    "dataset_config"    : "ARC-Challenge",
    "eval_split"        : "test",
    "train_split"       : "train",
    "eval_seed"         : 42,
    "eval_n"            : 900,
    "max_train_samples" : 1000,    # ARC-Challenge train ~1119 examples
    "lora_r"            : 16,
    "lora_alpha"        : 32,
    "lora_dropout"      : 0.05,
    "learning_rate"     : 2e-4,
    "num_epochs"        : 5,       # more epochs — smaller train set
    "batch_size"        : 4,
    "grad_accum"        : 4,
    "max_seq_length"    : 320,
    "max_new_tokens"    : 8,       # just a letter
    "results_file"      : f"{OUTPUT_DIR}/results.jsonl",
    "checkpoint_file"   : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"        : 100,
}
print("Config ready. eval_seed=42, eval_n=900.")
print("NOTE: ARC train set is small (~1119). eval N=900 comes from the test split.")

Config ready. eval_seed=42, eval_n=900.
NOTE: ARC train set is small (~1119). eval N=900 comes from the test split.


In [5]:
# CELL 5 — Load ARC-Challenge
# ARC-Challenge has train (~1119), validation (~299), test (~1172) splits.
# The paper eval uses N=900 from the test split with seed=42.
# Training uses the train split (no overlap with test).
print("Loading ARC-Challenge...")
raw_ds = load_dataset(CONFIG["dataset_name"], CONFIG["dataset_config"])
print(f"Splits: {list(raw_ds.keys())}")
for k in raw_ds: print(f"  {k}: {len(raw_ds[k])} examples")

def normalise_arc(item):
    q = item["question"].strip()
    choices = item["choices"]

    labels = choices["label"]
    texts  = choices["text"]

    alpha = [chr(ord('A') + i) for i in range(len(texts))]

    label_map = {orig: alpha[i] for i, orig in enumerate(labels)}

    choice_str = "\n".join(f"{alpha[i]}. {texts[i]}" for i in range(len(texts)))
    full_q = f"{q}\n\n{choice_str}"

    raw_ans = item["answerKey"].strip()
    answer  = label_map.get(raw_ans, raw_ans)

    if answer not in alpha:
        answer = raw_ans

    return {"question": full_q, "answer": answer}

eval_pool  = [normalise_arc(x) for x in raw_ds[CONFIG["eval_split"]]]
train_pool = [normalise_arc(x) for x in raw_ds[CONFIG["train_split"]]]
# Also add validation to train (more data helps with small train set)
val_pool   = [normalise_arc(x) for x in raw_ds["validation"]]

print(f"\nEval pool  (test)      : {len(eval_pool)}")
print(f"Train pool (train)     : {len(train_pool)}")
print(f"Val pool   (validation): {len(val_pool)}")
ad = Counter(x["answer"] for x in eval_pool)
print(f"Answer dist (eval): {dict(sorted(ad.items()))}")

Loading ARC-Challenge...


README.md: 0.00B [00:00, ?B/s]

ARC-Challenge/train-00000-of-00001.parqu(…):   0%|          | 0.00/190k [00:00<?, ?B/s]

ARC-Challenge/test-00000-of-00001.parque(…):   0%|          | 0.00/204k [00:00<?, ?B/s]

ARC-Challenge/validation-00000-of-00001.(…):   0%|          | 0.00/55.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1119 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1172 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/299 [00:00<?, ? examples/s]

Splits: ['train', 'test', 'validation']
  train: 1119 examples
  test: 1172 examples
  validation: 299 examples

Eval pool  (test)      : 1172
Train pool (train)     : 1119
Val pool   (validation): 299
Answer dist (eval): {'A': 266, 'B': 311, 'C': 310, 'D': 285}


In [6]:
# CELL 6 — Eval / train split
# Eval: N=900 from test split, seed=42 — matches paper
random.seed(CONFIG["eval_seed"])
if CONFIG["eval_n"] <= len(eval_pool):
    eval_data = random.sample(eval_pool, CONFIG["eval_n"])
else:
    eval_data = eval_pool
    print(f"WARNING: eval pool has only {len(eval_pool)} — using all")

eval_questions = set(x["question"] for x in eval_data)

# Train: train split + validation split (no overlap with test)
all_train = train_pool + val_pool
train_candidates = [x for x in all_train
                    if x["question"] not in eval_questions]
random.seed(0)
if len(train_candidates) > CONFIG["max_train_samples"]:
    train_data = random.sample(train_candidates, CONFIG["max_train_samples"])
else:
    train_data = train_candidates

overlap = eval_questions & set(x["question"] for x in train_data)
print(f"Eval  : {len(eval_data)} questions (seed=42, test split)")
print(f"Train : {len(train_data)} questions (train+val splits)")
print(f"Overlap: {len(overlap)} (must be 0)")

ed = Counter(x["answer"] for x in eval_data)
print(f"Eval answer dist: {dict(sorted(ed.items()))}")

Eval  : 900 questions (seed=42, test split)
Train : 1000 questions (train+val splits)
Overlap: 0 (must be 0)
Eval answer dist: {'A': 195, 'B': 255, 'C': 226, 'D': 224}


In [7]:
# CELL 7 — SFT prompt format
# Model trained to output only the answer letter (A/B/C/D).
# ARC is Tier-2: science knowledge is directly accessible.
# FT Solo should benefit most here — the 3B model knows science facts.

SYSTEM_PROMPT = (
    "You are a precise science knowledge assistant.\n"
    "Read the question and all options carefully.\n"
    "Respond with only the letter of the correct answer: A, B, C, or D."
)

def format_sft(item, tokenizer):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"Question:\n{item['question']}"},
        {"role": "assistant", "content": item["answer"]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

print("SFT format ready.")
print(f"Example:")
print(f"{train_data[0]['question']}")
print(f"Answer: {train_data[0]['answer']}")

SFT format ready.
Example:
Which of the following questions is testable in a scientific investigation?

A. Are dogs better pets than cats?.
B. Are dogs happy when they are walked?.
C. Are cats more active at night than during the day?.
D. Are cats easier to take care of than dogs?.
Answer: C


In [8]:
# CELL 8 — Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Tokenizer ready.")

config.json: 0.00B [00:00, ?B/s]

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Tokenizer ready.


In [9]:
# CELL 9 — Prepare HF Dataset
train_formatted = [format_sft(x, tokenizer) for x in train_data]
hf_train = Dataset.from_list(train_formatted)
print(f"Training examples: {len(hf_train)}")
print(f"Sample (first 280 chars):\n{hf_train[0]["text"][:280]}")

Training examples: 1000
Sample (first 280 chars):
<|system|>
You are a precise science knowledge assistant.
Read the question and all options carefully.
Respond with only the letter of the correct answer: A, B, C, or D.<|end|>
<|user|>
Question:
Which of the following questions is testable in a scientific investigation?

A. Are 


In [10]:
# CELL 10 — Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"], torch_dtype=torch.float16, device_map="auto"
)
base_model.config.use_cache = False
base_model.enable_input_require_grads()
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

VRAM: 3.82GB


In [11]:
# CELL 11 — Attach LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    bias="none",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 3,145,728 || all params: 3,824,225,280 || trainable%: 0.0823


In [12]:
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

In [13]:
# CELL 12 — Fine-tune
# ~15-20 min on T4. ARC train set is small so 5 epochs.


training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/checkpoints",
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=1,        # ← down from 4
    gradient_accumulation_steps=16,       # ← up from 4 (keeps effective batch=16)
    learning_rate=CONFIG["learning_rate"],
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    optim="adafactor",                    # ← saves ~2GB vs Adam
    logging_steps=100,
    save_strategy="epoch",
    report_to="none",
    gradient_checkpointing=True,          # ← enable here too
    dataloader_num_workers=0,
    seed=42,
)




import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()


trainer = SFTTrainer(
    model=model, train_dataset=hf_train,
    processing_class=tokenizer, args=training_args,
)
t0 = time.time()
trainer.train()
print(f"Training done in {(time.time()-t0)/60:.1f} min.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Step,Training Loss
100,0.886213
200,0.548633
300,0.510224


Training done in 21.3 min.


In [14]:
# CELL 13 — Save model
ft_model_path = f"{OUTPUT_DIR}/ft_model"
trainer.save_model(ft_model_path)
tokenizer.save_pretrained(ft_model_path)
print(f"Saved to {ft_model_path}")

Saved to /content/arc_ft3b_solo/ft_model


In [15]:
# CELL 14 — Answer extraction (A/B/C/D)
def extract_abcd_answer(text):
    text = text.strip()
    # 1. Single letter on first line
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    if lines:
        m = re.match(r"^\**([A-D])\**[.):,]?$", lines[0], re.IGNORECASE)
        if m: return m.group(1).upper()
    # 2. 'answer is X' or 'answer: X'
    m = re.search(
        r"(?:answer\s+is|answer:|correct\s+answer|therefore)\s*\**([A-D])\**",
        text, re.IGNORECASE)
    if m: return m.group(1).upper()
    # 3. Bold letter
    m = re.search(r"\*\*([A-D])\*\*", text)
    if m: return m.group(1).upper()
    # 4. Parenthesised
    m = re.search(r"\(([A-D])\)", text)
    if m: return m.group(1).upper()
    # 5. Any standalone A-D
    m = re.search(r"\b([A-D])\b", text)
    if m: return m.group(1).upper()
    return ""

_t = ["A","**B**","answer is C","(D)","The correct answer: B"]
_e = ["A","B","C","D","B"]
ok = all(extract_abcd_answer(t)==e for t,e in zip(_t,_e))
print("Extractor:", "PASSED" if ok else "FAIL")

Extractor: PASSED


In [16]:
# CELL 15 — Reload model for eval
del model, base_model, trainer
torch.cuda.empty_cache()

eval_base = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"], torch_dtype=torch.float16, device_map="auto"
).eval()
ft_model = PeftModel.from_pretrained(eval_base, ft_model_path).eval()
print("Fine-tuned model loaded for eval.")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Fine-tuned model loaded for eval.
VRAM: 7.67GB


In [17]:
# CELL 16 — Eval function (single greedy pass = 3.0B pp)
EVAL_SYSTEM = (
    "You are a precise science knowledge assistant.\n"
    "Read the question and all options carefully.\n"
    "Respond with only the letter of the correct answer: A, B, C, or D."
)

def run_ft_solo(question):
    messages = [
        {"role": "system", "content": EVAL_SYSTEM},
        {"role": "user",   "content": f"Question:\n{question}"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt",
                       truncation=True, max_length=512)
    device = next(ft_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_toks, skip_special_tokens=True).strip()

# Verification: 20 questions
v_correct = 0; v_empty = 0
for item in eval_data[:20]:
    raw  = run_ft_solo(item["question"])
    pred = extract_abcd_answer(raw)
    if not pred: v_empty += 1
    if pred == item["answer"]: v_correct += 1
print(f"Verification (20 q): {v_correct}/20 = {v_correct/20*100:.0f}%")
print(f"Empty: {v_empty}/20")

Verification (20 q): 16/20 = 80%
Empty: 0/20


In [18]:
# CELL 17 — Full evaluation N=900
# ~10-15 min on T4
print(f"Evaluating {CONFIG['eval_n']} questions | compute: 3.0B pp per question")
print("-"*60)

results = []; start_idx = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f: ck = json.load(f)
    start_idx = ck.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            results = [json.loads(l) for l in f if l.strip()]
    print(f"Resumed from {start_idx}")

t0 = time.time()
for idx in tqdm(range(start_idx, len(eval_data)), desc="ARC-FT3B"):
    item = eval_data[idx]
    try:
        raw  = run_ft_solo(item["question"])
        pred = extract_abcd_answer(raw)
        gt   = item["answer"]
        results.append({
            "idx"          : idx,
            "question"     : item["question"],
            "gt_answer"    : gt,
            "raw_output"   : raw,
            "final_answer" : pred,
            "correct"      : (pred == gt),
            "empty"        : (pred == ""),
        })
    except Exception as e:
        results.append({
            "idx": idx, "question": item["question"],
            "gt_answer": item["answer"], "raw_output": "",
            "final_answer": "", "correct": False,
            "empty": True, "error": str(e)
        })

    if (idx+1) % CONFIG["save_every"] == 0:
        with open(CONFIG["results_file"], "w") as f:
            for r in results: f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx+1}, f)
        acc  = sum(r["correct"] for r in results)/len(results)*100
        mins = (time.time()-t0)/60
        print(f"  [{idx+1:4d}] acc={acc:.1f}%  ({mins:.1f}min)")

with open(CONFIG["results_file"], "w") as f:
    for r in results: f.write(json.dumps(r) + "\n")

n_correct = sum(r["correct"] for r in results)
n_empty   = sum(r["empty"]   for r in results)
print(f"\nAccuracy : {n_correct}/{len(results)} = {n_correct/len(results)*100:.1f}%")
print(f"Empty    : {n_empty}")

Evaluating 900 questions | compute: 3.0B pp per question
------------------------------------------------------------


ARC-FT3B:   0%|          | 0/900 [00:00<?, ?it/s]

  [ 100] acc=81.0%  (0.2min)
  [ 200] acc=82.0%  (0.4min)
  [ 300] acc=83.0%  (0.6min)
  [ 400] acc=82.0%  (0.9min)
  [ 500] acc=83.4%  (1.1min)
  [ 600] acc=83.5%  (1.3min)
  [ 700] acc=83.3%  (1.5min)
  [ 800] acc=83.1%  (1.7min)
  [ 900] acc=84.2%  (1.9min)

Accuracy : 758/900 = 84.2%
Empty    : 0


In [19]:
# CELL 18 — Tier-2 validation check
# ARC is Tier-2: both CoT and Guided achieve 80.0%.
# Does FT Solo reach 80% at 3.5x less compute?
# Does it exceed 80% — beating both CoT and guided?

ft_acc = sum(r["correct"] for r in results) / len(results) * 100
pred_dist = Counter(r["final_answer"] for r in results)

print("Option selection rates (FT 3B Solo):")
for opt in ["A","B","C","D",""]:
    label = "(empty)" if opt == "" else opt
    n = pred_dist.get(opt, 0)
    print(f"  {label}: {n} ({n/len(results)*100:.1f}%)  [expected ~25%]")

# ECE calculation
# Single-pass greedy: confidence is always 1.0 (or 0 for empty)
# So ECE = |accuracy - 1.0| for non-empty predictions
non_empty_correct = sum(1 for r in results if r["correct"] and not r["empty"])
non_empty = sum(1 for r in results if not r["empty"])
if non_empty > 0:
    hc_acc = non_empty_correct / non_empty
    print(f"\nGreedy-pass accuracy on non-empty: {hc_acc*100:.1f}%")
    print(f"(All predictions are 100% confident — ECE = |acc - 1.0| = {1-hc_acc:.4f})")

Option selection rates (FT 3B Solo):
  A: 187 (20.8%)  [expected ~25%]
  B: 260 (28.9%)  [expected ~25%]
  C: 245 (27.2%)  [expected ~25%]
  D: 208 (23.1%)  [expected ~25%]
  (empty): 0 (0.0%)  [expected ~25%]

Greedy-pass accuracy on non-empty: 84.2%
(All predictions are 100% confident — ECE = |acc - 1.0| = 0.1578)


In [20]:
# CELL 19 — Final comparison table
ft_acc = sum(r["correct"] for r in results) / len(results) * 100

# Paper confirmed values for ARC-Challenge
BASELINE = 71.7
COT      = 80.0
GUIDED   = 80.0
BASE_3B_ABL = 74.0   # untuned 3B ablation (paper Table XIII)

print("="*70)
print("ARC-Challenge — FULL COMPUTE-ACCURACY COMPARISON")
print("="*70)
print(f"  Condition                | Compute   | Accuracy | Note")
print(f"  -------------------------|-----------|----------|-----")
print(f"  Baseline (1.5B×5)        | 7.5B pp   | {BASELINE}%  |")
print(f"  Base-3B ablation (untuned)| 10.5B pp | {BASE_3B_ABL}%  | paper Table XIII")
print(f"  CoT (1.5B×5)             | 7.5B pp   | {COT}%  | Tier-2: CoT sufficient")
print(f"  FT 3B Solo (this run)    | 3.0B pp   | {ft_acc:.1f}%  |")
print(f"  Guided pipeline          | 10.5B pp  | {GUIDED}%  | Tier-2: CoT = Guided")
print()
gap_vs_guided  = GUIDED   - ft_acc
gap_vs_cot     = COT      - ft_acc
gap_vs_base    = ft_acc   - BASELINE
gap_vs_base3b  = ft_acc   - BASE_3B_ABL
print(f"  FT Solo vs Baseline      : {gap_vs_base:+.1f} pts  (at 2.5x less compute)")
print(f"  FT Solo vs Base-3B abl.  : {gap_vs_base3b:+.1f} pts")
print(f"  FT Solo vs CoT           : {-gap_vs_cot:+.1f} pts  (both 7.5B vs 3.0B)")
print(f"  FT Solo vs Guided        : {-gap_vs_guided:+.1f} pts  (guided costs 3.5x more)")
print()
if ft_acc >= 80.0:
    print("  VERDICT: FT Solo matches or exceeds CoT and Guided.")
    print("  On ARC (Tier-2), fine-tuning the 3B model directly is the")
    print("  compute-optimal strategy — same accuracy at 3.5x lower cost.")
elif ft_acc >= 75.0:
    print(f"  VERDICT: FT Solo close to CoT/Guided ({gap_vs_cot:.1f} pts behind).")
    print("  Fine-tuning helps but does not fully close the gap with CoT.")
else:
    print(f"  VERDICT: FT Solo trails CoT by {-gap_vs_cot:.1f} pts despite fine-tuning.")
    print("  Surprising for Tier-2 — investigate whether label remapping")
    print("  (1/2/3/4 -> A/B/C/D) caused any mismatch in Cell 5.")

ARC-Challenge — FULL COMPUTE-ACCURACY COMPARISON
  Condition                | Compute   | Accuracy | Note
  -------------------------|-----------|----------|-----
  Baseline (1.5B×5)        | 7.5B pp   | 71.7%  |
  Base-3B ablation (untuned)| 10.5B pp | 74.0%  | paper Table XIII
  CoT (1.5B×5)             | 7.5B pp   | 80.0%  | Tier-2: CoT sufficient
  FT 3B Solo (this run)    | 3.0B pp   | 84.2%  |
  Guided pipeline          | 10.5B pp  | 80.0%  | Tier-2: CoT = Guided

  FT Solo vs Baseline      : +12.5 pts  (at 2.5x less compute)
  FT Solo vs Base-3B abl.  : +10.2 pts
  FT Solo vs CoT           : +4.2 pts  (both 7.5B vs 3.0B)
  FT Solo vs Guided        : +4.2 pts  (guided costs 3.5x more)

  VERDICT: FT Solo matches or exceeds CoT and Guided.
  On ARC (Tier-2), fine-tuning the 3B model directly is the
  compute-optimal strategy — same accuracy at 3.5x lower cost.
